In [0]:
dbutils.widgets.text("catalog", "dev_l2")
catalog = dbutils.widgets.get("catalog")
spark.sql(f"USE catalog {catalog}")

from pyspark.sql.types import StructType, StructField, StringType, LongType

table_configs = [
    {"name": "region",   "pk": ["r_regionkey"]},
    {"name": "nation",   "pk": ["n_nationkey"]},
    {"name": "customer", "pk": ["c_custkey"]},
    {"name": "supplier", "pk": ["s_suppkey"]},
    {"name": "order",    "pk": ["o_orderkey"]},
    {"name": "part",     "pk": ["p_partkey"]},
    {"name": "partsupp", "pk": ["ps_partkey", "ps_suppkey"]},
    {"name": "lineitem", "pk": ["l_orderkey", "l_linenumber"]},
]

gold_mappings = {
    "lineitem": "gold.fact_sales",
    "customer": "gold.dim_customer",
    "part":     "gold.dim_product",
    "supplier": "gold.dim_supplier",
    "nation":   "gold.dim_geography",
}

dq_failures = 0

In [0]:
# CHECK 1: All 8 bronze tables ingested (exist and have rows > 0)
results = []
for cfg in table_configs:
    table = cfg["name"]
    try:
        row_count = spark.sql(f"SELECT COUNT(*) FROM bronze.{table}").collect()[0][0]
        status = "PASS" if row_count > 0 else "FAIL"
        if status == "FAIL":
            dq_failures += 1
        results.append((f"bronze.{table}", row_count, status))
    except Exception:
        dq_failures += 1
        results.append((f"bronze.{table}", 0, "FAIL"))

display(spark.createDataFrame(results, StructType([
    StructField("table_name", StringType(), True),
    StructField("row_count", LongType(), True),
    StructField("status", StringType(), True),
])))

In [0]:
# CHECK 2: Primary keys populated (non-null) and unique in silver tables
results = []
for cfg in table_configs:
    table = cfg["name"]
    pk_cols = cfg["pk"]
    pk_str = ", ".join(pk_cols)

    total = spark.sql(f"SELECT COUNT(*) FROM silver.{table}").collect()[0][0]
    distinct = spark.sql(f"SELECT COUNT(*) FROM (SELECT DISTINCT {pk_str} FROM silver.{table})").collect()[0][0]
    status = "PASS" if total == distinct else "FAIL"
    if status == "FAIL":
        dq_failures += 1
    results.append((f"silver.{table}", total, distinct, status))

display(spark.createDataFrame(results, StructType([
    StructField("table_name", StringType(), True),
    StructField("total_rows", LongType(), True),
    StructField("distinct_rows", LongType(), True),
    StructField("status", StringType(), True),
])))

In [0]:
# CHECK 3: Gold revenue and row counts reconcile with silver
results = []

silver_revenue = spark.sql("SELECT COALESCE(SUM(l_extendedprice * (1 - l_discount)), 0) FROM silver.lineitem").collect()[0][0]
gold_revenue = spark.sql("SELECT COALESCE(SUM(net_price), 0) FROM gold.fact_sales").collect()[0][0]
rev_pass = abs(float(silver_revenue) - float(gold_revenue)) < 0.01
if not rev_pass:
    dq_failures += 1
results.append(("revenue: lineitem_vs_fact_sales", int(float(silver_revenue)), int(float(gold_revenue)), "PASS" if rev_pass else "FAIL"))

for silver_table, gold_table in gold_mappings.items():
    silver_count = spark.sql(f"SELECT COUNT(*) FROM silver.{silver_table}").collect()[0][0]
    gold_count = spark.sql(f"SELECT COUNT(*) FROM {gold_table}").collect()[0][0]
    status = "PASS" if silver_count == gold_count else "FAIL"
    if status == "FAIL":
        dq_failures += 1
    results.append((f"{silver_table} vs {gold_table.split('.')[-1]}", silver_count, gold_count, status))

display(spark.createDataFrame(results, StructType([
    StructField("table_name", StringType(), True),
    StructField("silver_count", LongType(), True),
    StructField("gold_count", LongType(), True),
    StructField("status", StringType(), True),
])))

In [0]:
# CHECK 4: Bronze row counts reconcile with silver + quarantine
results = []
for cfg in table_configs:
    table = cfg["name"]
    bronze_count = spark.sql(f"SELECT COUNT(*) FROM bronze.{table}").collect()[0][0]
    silver_count = spark.sql(f"SELECT COUNT(*) FROM silver.{table}").collect()[0][0]
    try:
        quarantine_count = spark.sql(f"SELECT COUNT(*) FROM silver.quarantine_{table}").collect()[0][0]
    except Exception:
        quarantine_count = 0
    status = "PASS" if bronze_count == silver_count + quarantine_count else "FAIL"
    if status == "FAIL":
        dq_failures += 1
    results.append((table, bronze_count, silver_count, quarantine_count, status))

display(spark.createDataFrame(results, StructType([
    StructField("table_name", StringType(), True),
    StructField("bronze_count", LongType(), True),
    StructField("silver_count", LongType(), True),
    StructField("quarantine_count", LongType(), True),
    StructField("status", StringType(), True),
])))

In [0]:
# Halt pipeline if any DQ check failed
if dq_failures > 0:
    raise RuntimeError(f"Data quality check FAILED: {dq_failures} critical violation(s) detected. Pipeline halted.")